# ADK: Prompt Optimizer & GEPA (March 2026 Suite)

[![Open In Colab](https://colab.research.google.com/github/maruti123/partner-demos/blob/main/partner-demos-march-2026/adk_prompt_optimizer_demo.ipynb)](https://colab.research.google.com/github/maruti123/partner-demos/blob/main/partner-demos-march-2026/adk_prompt_optimizer_demo.ipynb)

**What you'll see:** The ADK Optimizer (`adk optimize`) systematically improves an agent's instructions using a teacher model and ground-truth evaluation data — no manual prompt tweaking.

## Scenario: From 75% to Production-Ready

A partner's "Sales Support" agent answers product questions but only scores ~75% against ground truth. Instead of manually rewriting prompts, they use GEPA:

| Step | What Happens |
|------|-------------|
| 1 | **Define ground truth** — Create an eval set with questions and expected answers |
| 2 | **Write the agent** — Standard ADK agent directory with a baseline instruction |
| 3 | **Run `adk optimize`** — GEPA iteratively tests, scores, and refines the instruction |
| 4 | **Compare** — Run both baseline and optimized agents against eval data |

### Key Technologies
- **[GEPA](https://github.com/gepa-ai/gepa)** — LLM-based reflection and Pareto-efficient evolutionary search for optimizing prompts
- **`adk optimize`** — CLI command that runs the GEPA pipeline end-to-end via ADK's adapter
- **google-adk v1.28.0+** — Agent framework with built-in optimization support

### Two Models in Play

| Role | Model | Purpose |
|------|-------|---------|
| **Agent** (being optimized) | Gemini 3.1 Pro (Preview) | Answers product questions — this is the prompt we're improving |
| **Teacher** (GEPA optimizer) | Gemini 2.5 Flash (ADK default) | Analyzes agent failures and proposes better instructions. Uses thinking (`budget_tokens=10240`) for deeper reflection |

> The teacher model is configured in `GEPARootAgentPromptOptimizerConfig`. Override it via `--optimizer_config_file_path` if needed.

### Requirements
- `google-adk[eval] >= 1.28.0` installed (includes GEPA dependency)
- Access to Gemini models via Vertex AI
- A Google Cloud project with Vertex AI API enabled

In [ ]:
# 1. Setup and Authentication
%pip install "google-adk[eval]>=1.28.0" google-genai nest-asyncio --quiet --index-url https://pypi.org/simple

try:
    from google.colab import auth
    auth.authenticate_user()
    print('Authenticated via Colab')
except ModuleNotFoundError:
    print('Not running in Colab — using Application Default Credentials (ADC)')

import os
import nest_asyncio
nest_asyncio.apply()

project_id = 'iamtests-315719'  # @param {type:"string"}
location = 'global'  # @param {type:"string"} — Gemini 3.1 Pro Preview requires 'global'
os.environ["GOOGLE_CLOUD_PROJECT"] = project_id
os.environ["GOOGLE_CLOUD_LOCATION"] = location
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"

### 2. Create the Agent Directory

`adk optimize` expects a standard ADK agent directory on disk. We use `%%writefile` to create it inline so the notebook is fully self-contained.

In [ ]:
import json
import os
import time
import uuid

agent_dir = "sales_support_agent"
baseline_instruction = "You are a sales support agent. Answer product questions."

os.makedirs(agent_dir, exist_ok=True)

with open(os.path.join(agent_dir, "__init__.py"), "w") as f:
    f.write("from .agent import agent\n")

with open(os.path.join(agent_dir, "agent.py"), "w") as f:
    f.write(f'''from google.adk import Agent

agent = Agent(
    model="gemini-3.1-pro-preview",
    name="SalesSupport",
    instruction="{baseline_instruction}"
)
''')

# Eval set — must be <eval_set_id>.evalset.json inside the agent directory
eval_pairs = [
    ("What is the warranty period for the ProCamera X1?",
     "The ProCamera X1 comes with a 24-month standard international warranty."),
    ("Does the Tripod Lite support 4K video cameras?",
     "Yes, the Tripod Lite supports cameras up to 5kg, including 4K DSLR models."),
    ("What is the return policy for the LensPro 50mm?",
     "The LensPro 50mm has a 30-day return policy with full refund if unopened."),
    ("Is the FlashKit Pro compatible with third-party cameras?",
     "Yes, the FlashKit Pro is compatible with Canon, Nikon, Sony, and Fujifilm mounts."),
    ("What is the battery life of the ProCamera X1?",
     "The ProCamera X1 battery lasts approximately 800 shots per charge under standard use."),
]

eval_set = {
    "eval_set_id": "train_eval_set",
    "name": "train_eval_set",
    "eval_cases": [
        {
            "eval_id": f"case_{i}",
            "conversation": [{
                "invocation_id": str(uuid.uuid4()),
                "user_content": {"parts": [{"text": query}], "role": "user"},
                "final_response": {"parts": [{"text": reference}], "role": "model"},
            }],
            "session_input": {"app_name": agent_dir, "user_id": "user"},
            "creation_timestamp": time.time(),
        }
        for i, (query, reference) in enumerate(eval_pairs)
    ],
    "creation_timestamp": time.time(),
}

eval_set_path = os.path.join(agent_dir, "train_eval_set.evalset.json")
with open(eval_set_path, "w") as f:
    json.dump(eval_set, f, indent=2)

# Sampler config — tells adk optimize which eval set to use and how to score
sampler_config = {
    "eval_config": {
        "criteria": {
            "response_match_score": 0.5
        }
    },
    "app_name": agent_dir,
    "train_eval_set": "train_eval_set",
}

sampler_config_path = os.path.join(agent_dir, "sampler_config.json")
with open(sampler_config_path, "w") as f:
    json.dump(sampler_config, f, indent=2)

print(f"Agent directory created: {agent_dir}/")
print(f"  agent.py — baseline instruction")
print(f"  __init__.py — ADK discovery")
print(f"  train_eval_set.evalset.json — {len(eval_pairs)} eval cases")
print(f"  sampler_config.json — response_match_score >= 0.5")

### 3. Run `adk optimize` (GEPA)

This is the core feature. `adk optimize` reads the agent directory, evaluates the baseline instruction against the eval set, then uses GEPA's evolutionary search to generate improved instructions.

```bash
adk optimize sales_support_agent \
  --sampler_config_file_path sales_support_agent/sampler_config.json \
  --print_detailed_results
```

> **How GEPA works:** It uses LLM-based reflection to analyze agent failures, then applies Pareto-efficient evolutionary search to explore the prompt space — generating candidate instructions, scoring them against eval data, and iteratively selecting the best-performing variants.

In [ ]:
# Run adk optimize — GEPA iteratively tests, scores, and refines the instruction
# This usually takes 2-5 minutes depending on the number of eval cases.
result = !adk optimize {agent_dir} --sampler_config_file_path {agent_dir}/sampler_config.json --print_detailed_results

print("\n".join(result))

# Auto-extract the optimized instruction from output
optimized_instruction = None
for i, line in enumerate(result):
    if "Optimized root agent instructions:" in line:
        optimized_instruction = "\n".join(result[i+1:]).strip()
        break

if optimized_instruction:
    print(f"\n✓ Captured optimized instruction ({len(optimized_instruction)} chars)")
else:
    print("\nNote: Could not auto-extract — paste it into the next cell's fallback variable.")

### 4. Compare Baseline vs. Optimized Agent

Now we run both agents against the same eval questions to see the improvement. The optimized agent uses the instruction generated by GEPA in the previous step.

> **If `adk optimize` output format differs:** Copy the optimized instruction from the output above and paste it into the `optimized_instruction` variable in the fallback section below.

In [ ]:
from google.adk import Agent, Runner
from google.adk.sessions.in_memory_session_service import InMemorySessionService
from google.genai import types

if not optimized_instruction:
    optimized_instruction = """You are a technical sales support agent specializing in camera
    equipment. Always provide specific product details: warranty type (International vs Local),
    weight limits, compatibility lists, return windows, and battery specifications.
    Reference the 2026 Master Catalog for accuracy. When comparing products, include
    price tier and availability region."""
    print("Using fallback instruction — paste GEPA output into optimized_instruction for real results.\n")

baseline_agent = Agent(
    model="gemini-3.1-pro-preview",
    name="SalesSupport",
    instruction=baseline_instruction
)

optimized_agent = Agent(
    model="gemini-3.1-pro-preview",
    name="SalesSupportOptimized",
    instruction=optimized_instruction
)

eval_questions = [
    "What is the warranty period for the ProCamera X1?",
    "Does the Tripod Lite support 4K video cameras?",
    "What is the battery life of the ProCamera X1?",
]

async def compare_agents():
    for label, agent in [("BASELINE", baseline_agent), ("OPTIMIZED", optimized_agent)]:
        runner = Runner(
            agent=agent,
            session_service=InMemorySessionService(),
            app_name="prompt_optimizer_demo",
            auto_create_session=True
        )
        print(f"\n{'='*60}")
        print(f"  {label} Agent")
        print(f"  Instruction: {agent.instruction[:80]}...")
        print(f"{'='*60}")

        for i, q in enumerate(eval_questions):
            print(f"\nQ: {q}")
            message = types.Content(parts=[types.Part(text=q)], role='user')
            async for event in runner.run_async(
                user_id="eval_user",
                session_id=f"{label.lower()}_eval_{i}",
                new_message=message
            ):
                if event.content and event.content.parts:
                    for part in event.content.parts:
                        if part.text:
                            print(f"A: {part.text}")

await compare_agents()

### 5. Key Takeaways

| Feature | What It Does | Partner Value |
|---------|-------------|---------------|
| **`adk optimize`** | Runs the GEPA pipeline end-to-end from CLI | One command to improve any agent's instructions |
| **GEPA** | Evolutionary search with LLM-based reflection | Measurable prompt improvement, not guesswork |
| **Two models** | Agent (3.1 Pro) + Teacher (2.5 Flash with thinking) | Teacher analyzes failures cheaply; agent runs at full capability |
| **Eval Set** | Ground-truth Q&A pairs in `.evalset.json` format | Quality of optimization depends on quality of eval data |
| **Runner Pattern** | Manages sessions and streams agent responses | Standard pattern across all March 2026 demos |

#### How GEPA Compares to Manual Prompt Engineering

| | GEPA (`adk optimize`) | Manual Prompt Tweaking |
|---|---|---|
| **Process** | Automated: evaluate → reflect on failures → mutate → accept/reject → repeat | Ad-hoc: write → test → guess → rewrite |
| **Measurability** | Scores against ground truth each iteration | Subjective "does it feel better?" |
| **Scalability** | Works across multi-agent orchestrations | One agent at a time |
| **Reproducibility** | Deterministic given same eval data | Depends on the engineer's intuition |

> **Production tip:** Invest in high-quality eval datasets. GEPA is only as good as the ground truth you give it. 50+ diverse Q&A pairs is a good starting point.

### 6. Cleanup (Optional)

Remove the agent directory created during the demo.

In [ ]:
# Uncomment and run to delete the agent directory
# import shutil
# shutil.rmtree("sales_support_agent", ignore_errors=True)
# print("Cleaned up sales_support_agent/ directory.")